# MCF Job Matching — Analytics & ML Exploration

Three analyses:
1. **Cluster Structure Discovery** — find natural clusters in the job embedding space
2. **Cluster vs Category Salary Accuracy** — which grouping gives tighter pay bands?
3. **Classification Model** — can embeddings predict MCF labels? (validates embedding quality)

### Prerequisites
```
pip install jupyter matplotlib pandas seaborn
```
numpy, scikit-learn, scipy, duckdb are already in requirements.txt.

In [1]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score, classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import json
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 100

In [2]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Set MCF_DB_PATH env var, or edit this directly.
# The .duckdb file is typically at data/mcf.duckdb relative to the project root.
DB_PATH = os.getenv('MCF_DB_PATH', '../data/mcf.duckdb')

print(f'Connecting to: {DB_PATH}')
con = duckdb.connect(DB_PATH, read_only=True)

# Load active jobs with embeddings + metadata
df = con.execute("""
    SELECT
        j.job_uuid,
        j.title,
        j.salary_min,
        j.salary_max,
        COALESCE(NULLIF(TRIM(COALESCE(json_extract_string(j.categories_json, '$[0]'), '')), ''), 'Unknown')       AS category,
        COALESCE(NULLIF(TRIM(COALESCE(json_extract_string(j.employment_types_json, '$[0]'), '')), ''), 'Unknown') AS employment_type,
        COALESCE(NULLIF(TRIM(COALESCE(json_extract_string(j.position_levels_json, '$[0]'), '')), ''), 'Unknown')  AS position_level,
        je.embedding_json
    FROM jobs j
    JOIN job_embeddings je ON je.job_uuid = j.job_uuid
    WHERE j.is_active = TRUE
      AND (j.job_source = 'mcf' OR j.job_source IS NULL)
    LIMIT 8000
""").df()

# Parse JSON string embeddings → float32 numpy matrix
X = np.array([json.loads(e) for e in df['embedding_json']], dtype=np.float32)
df = df.drop(columns=['embedding_json'])  # free memory

print(f'Loaded {len(df):,} jobs | embedding dim = {X.shape[1]}')
print(f'Jobs with salary_min: {df["salary_min"].notna().sum():,} ({df["salary_min"].notna().mean()*100:.1f}%)')
print()
print('Top categories:', df['category'].value_counts().head(8).to_dict())
print('Position levels:', df['position_level'].value_counts().to_dict())

Connecting to: ../data/mcf.duckdb
Loaded 8,000 jobs | embedding dim = 384
Jobs with salary_min: 120 (1.5%)

Top categories: {'Unknown': 7880, 'Admin / Secretarial': 14, 'Building and Construction': 12, 'Accounting / Auditing / Taxation': 12, 'Information Technology': 11, 'Customer Service': 10, 'F&B': 7, 'Engineering': 7}
Position levels: {'Unknown': 7880, 'Executive': 35, 'Professional': 20, 'Junior Executive': 16, 'Fresh/entry level': 16, 'Non-executive': 12, 'Manager': 12, 'Senior Executive': 6, 'Middle Management': 2, 'Senior Management': 1}


---
## Analysis 1 — Cluster Structure Discovery

Find the natural number of semantic job clusters using the elbow method + silhouette score.
Then visualise the embedding space in 2D with PCA.

In [ ]:
# ── 1a. Elbow + Silhouette curve ───────────────────────────────────────────────
ks = list(range(5, 41, 5))
inertias, sil_scores = [], []

print('Computing K-Means for k =', ks, '...')
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=5, max_iter=200)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil = silhouette_score(X, labels, sample_size=min(3000, len(X)), random_state=42)
    sil_scores.append(sil)
    print(f'  k={k:2d}  inertia={km.inertia_:,.0f}  silhouette={sil:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ks, inertias, 'o-', color='steelblue')
ax1.set(title='Elbow Curve', xlabel='Number of clusters (k)', ylabel='Inertia')
ax2.plot(ks, sil_scores, 'o-', color='coral')
ax2.set(title='Silhouette Score', xlabel='Number of clusters (k)', ylabel='Score (higher = better)')
plt.tight_layout()
plt.show()

best_k = ks[int(np.argmax(sil_scores))]
print(f'\nBest k by silhouette: {best_k}')

In [ ]:
# ── 1b. Fit final K-Means at best_k ───────────────────────────────────────────
print(f'Fitting KMeans with k={best_k}...')
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10, max_iter=300)
df['cluster'] = km_final.fit_predict(X)

# Top-5 job titles per cluster
print('\nTop job titles per cluster:')
for c in range(best_k):
    titles = df[df['cluster'] == c]['title'].value_counts().head(5).index.tolist()
    print(f'  Cluster {c:2d}: {titles}')

In [ ]:
# ── 1c. PCA 2D visualisation ───────────────────────────────────────────────────
print('Running PCA(2D)...')
pca2 = PCA(n_components=2, random_state=42)
coords = pca2.fit_transform(X)
df['pca_x'] = coords[:, 0]
df['pca_y'] = coords[:, 1]
print(f'Explained variance ratio: {pca2.explained_variance_ratio_.sum()*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Coloured by cluster
scatter1 = axes[0].scatter(df['pca_x'], df['pca_y'], c=df['cluster'],
                            cmap='tab20', alpha=0.4, s=8)
axes[0].set(title=f'Job Embeddings (PCA 2D) — {best_k} Clusters', xlabel='PC1', ylabel='PC2')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Coloured by MCF category
top_cats = df['category'].value_counts().head(10).index
cat_map = {c: i for i, c in enumerate(top_cats)}
cat_colors = [cat_map.get(c, -1) for c in df['category']]
scatter2 = axes[1].scatter(df['pca_x'], df['pca_y'], c=cat_colors,
                            cmap='tab10', alpha=0.4, s=8)
axes[1].set(title='Job Embeddings (PCA 2D) — MCF Category', xlabel='PC1', ylabel='PC2')
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=plt.cm.tab10(i/10), markersize=8)
           for i in range(len(top_cats))]
axes[1].legend(handles, top_cats, fontsize=7, loc='lower right')
plt.tight_layout()
plt.show()

---
## Analysis 2 — Cluster vs Category Salary Accuracy

For jobs with salary data, compare salary band tightness (IQR) when grouped by:
- **Embedding cluster** (discovered above)
- **MCF category** (assigned by MyCareersFuture)

Lower IQR = tighter, more accurate pay estimate.

In [ ]:
# ── 2a. Salary stats by cluster and by category ────────────────────────────────
sal_df = df[df['salary_min'].notna()].copy()
print(f'Jobs with salary data: {len(sal_df):,}')

def group_salary_stats(df, col, min_count=5):
    stats = df.groupby(col)['salary_min'].agg(
        count='count',
        p25=lambda x: x.quantile(0.25),
        median='median',
        p75=lambda x: x.quantile(0.75),
    ).reset_index()
    stats['iqr'] = stats['p75'] - stats['p25']
    return stats[stats['count'] >= min_count].sort_values('median', ascending=False)

cluster_stats = group_salary_stats(sal_df, 'cluster')
category_stats = group_salary_stats(sal_df, 'category')

print(f'\nSalary IQR comparison:')
print(f'  Mean IQR by cluster:  ${cluster_stats["iqr"].mean():,.0f}')
print(f'  Mean IQR by category: ${category_stats["iqr"].mean():,.0f}')
print(f'  Median IQR by cluster:  ${cluster_stats["iqr"].median():,.0f}')
print(f'  Median IQR by category: ${category_stats["iqr"].median():,.0f}')
print()
print('Cluster salary bands (top 15 by median):')
print(cluster_stats.head(15).to_string(index=False))
print()
print('Category salary bands:')
print(category_stats.to_string(index=False))

In [ ]:
# ── 2b. IQR distribution comparison ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IQR histogram
axes[0].hist(cluster_stats['iqr'], bins=20, alpha=0.7, label=f'By cluster (k={best_k})', color='steelblue')
axes[0].hist(category_stats['iqr'], bins=10, alpha=0.7, label='By MCF category', color='coral')
axes[0].axvline(cluster_stats['iqr'].mean(), color='steelblue', linestyle='--', linewidth=2)
axes[0].axvline(category_stats['iqr'].mean(), color='coral', linestyle='--', linewidth=2)
axes[0].set(title='Salary IQR Distribution', xlabel='IQR (SGD)', ylabel='Count')
axes[0].legend()

# Median salary by cluster (top 20)
plot_data = cluster_stats.head(20)
axes[1].barh(range(len(plot_data)), plot_data['median'], xerr=[
    plot_data['median'] - plot_data['p25'],
    plot_data['p75'] - plot_data['median']
], color='steelblue', alpha=0.7, capsize=4)
axes[1].set_yticks(range(len(plot_data)))
axes[1].set_yticklabels([f'Cluster {c}' for c in plot_data['cluster']], fontsize=9)
axes[1].set(title='Median Salary by Cluster (top 20, ±IQR)', xlabel='Salary (SGD/month)')

plt.tight_layout()
plt.show()

# Interpretation
cluster_mean_iqr = cluster_stats['iqr'].mean()
category_mean_iqr = category_stats['iqr'].mean()
improvement = (category_mean_iqr - cluster_mean_iqr) / category_mean_iqr * 100
if cluster_mean_iqr < category_mean_iqr:
    print(f'✓ Clusters produce {improvement:.1f}% tighter salary bands than MCF categories.')
    print('  → Cluster-based pay estimation would be more accurate than category-based.')
else:
    print('✗ MCF categories produce tighter salary bands than embedding clusters.')
    print('  → Current category-based approach is already well-suited for pay estimation.')

---
## Analysis 3 — Classification Model

Train a logistic regression classifier on job embeddings to predict:
- MCF **category** (broad job domain)
- MCF **position level** (seniority)

High accuracy validates that embeddings encode classification-relevant features.
Also measures cluster purity — how well discovered clusters align with MCF taxonomy.

In [ ]:
# ── 3a. Logistic regression on category and position_level ────────────────────
for target in ['category', 'position_level']:
    mask = (df[target] != 'Unknown') & (df[target].notna())
    # Only keep classes with enough samples for stratified split
    counts = df[mask][target].value_counts()
    valid_classes = counts[counts >= 5].index
    mask2 = mask & df[target].isin(valid_classes)

    X_cls = X[mask2]
    le = LabelEncoder()
    y = le.fit_transform(df[target][mask2])

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_cls, y, test_size=0.2, stratify=y, random_state=42
    )

    clf = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', multi_class='multinomial')
    clf.fit(X_tr, y_tr)
    acc = clf.score(X_te, y_te)

    print(f'\n{'='*60}')
    print(f'Target: {target}  |  Classes: {len(le.classes_)}  |  Samples: {len(X_cls):,}')
    print(f'Test accuracy: {acc:.3f} ({acc*100:.1f}%)')
    print()
    print(classification_report(y_te, clf.predict(X_te), target_names=le.classes_, digits=2))

In [ ]:
# ── 3b. Confusion matrix for category ─────────────────────────────────────────
mask = (df['category'] != 'Unknown') & (df['category'].notna())
counts = df[mask]['category'].value_counts()
# Limit to top-12 categories for readability
top_n = 12
valid_classes = counts.head(top_n).index
mask2 = mask & df['category'].isin(valid_classes)

X_cls = X[mask2]
le = LabelEncoder()
y = le.fit_transform(df['category'][mask2])
X_tr, X_te, y_tr, y_te = train_test_split(X_cls, y, test_size=0.2, stratify=y, random_state=42)
clf_cat = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', multi_class='multinomial')
clf_cat.fit(X_tr, y_tr)

fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay.from_estimator(
    clf_cat, X_te, y_te,
    display_labels=le.classes_,
    xticks_rotation=45,
    normalize='true',
    ax=ax,
    colorbar=False,
)
ax.set_title(f'Confusion Matrix — Category Classification (top {top_n} categories, normalised)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3c. Cluster purity analysis ───────────────────────────────────────────────
# For each cluster, what fraction of jobs share the most common MCF category?
# High purity → clusters refine MCF categories
# Low purity  → clusters capture different structure (e.g. seniority, domain blend)

def cluster_purity(df, cluster_col, label_col):
    results = []
    for c in sorted(df[cluster_col].unique()):
        sub = df[df[cluster_col] == c]
        counts = sub[label_col].value_counts()
        top_label = counts.index[0]
        purity = counts.iloc[0] / len(sub)
        results.append({'cluster': c, 'size': len(sub), 'top_label': top_label, 'purity': purity})
    return pd.DataFrame(results)

cat_purity = cluster_purity(df, 'cluster', 'category')
pl_purity  = cluster_purity(df, 'cluster', 'position_level')

print('Cluster purity (category):')
print(f'  Mean: {cat_purity["purity"].mean():.3f} | Median: {cat_purity["purity"].median():.3f}')
print()
print('Cluster purity (position level):')
print(f'  Mean: {pl_purity["purity"].mean():.3f} | Median: {pl_purity["purity"].median():.3f}')

# Show each cluster with its dominant label and purity
summary = cat_purity.merge(pl_purity[['cluster', 'top_label', 'purity']], on='cluster', suffixes=('_cat', '_pl'))
summary.columns = ['cluster', 'size', 'top_category', 'cat_purity', 'top_position_level', 'pl_purity']
print()
print(summary.sort_values('cat_purity', ascending=False).to_string(index=False))

In [ ]:
# ── 3d. Purity visualisation ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cat_purity['purity'], bins=15, color='steelblue', alpha=0.8)
axes[0].axvline(cat_purity['purity'].mean(), color='red', linestyle='--', label=f'Mean {cat_purity["purity"].mean():.2f}')
axes[0].set(title='Cluster Purity — MCF Category', xlabel='Purity (fraction with dominant label)', ylabel='# Clusters')
axes[0].legend()

axes[1].hist(pl_purity['purity'], bins=15, color='coral', alpha=0.8)
axes[1].axvline(pl_purity['purity'].mean(), color='navy', linestyle='--', label=f'Mean {pl_purity["purity"].mean():.2f}')
axes[1].set(title='Cluster Purity — Position Level', xlabel='Purity (fraction with dominant label)', ylabel='# Clusters')
axes[1].legend()

plt.tight_layout()
plt.show()

# Final interpretation
print()
print('=== Summary ===')
print()

# Cluster IQR vs category IQR
cluster_iqr = group_salary_stats(sal_df, 'cluster')['iqr'].mean()
cat_iqr = group_salary_stats(sal_df, 'category')['iqr'].mean()
if cluster_iqr < cat_iqr:
    print(f'✓ Pay accuracy: clusters give {(cat_iqr - cluster_iqr)/cat_iqr*100:.1f}% tighter IQR than categories.')
    print('  Recommendation: implement cluster-based pay estimation in the lowball checker.')
else:
    print('✗ Pay accuracy: MCF categories are already tighter. No cluster-based pay improvement needed.')

# Classification accuracy hint
print()
mean_cat_purity = cat_purity['purity'].mean()
if mean_cat_purity > 0.6:
    print(f'✓ Cluster purity (category): {mean_cat_purity:.2f} — clusters largely refine MCF categories.')
    print('  Recommendation: embedding clusters are a fine-grained version of MCF categories.')
elif mean_cat_purity > 0.4:
    print(f'~ Cluster purity (category): {mean_cat_purity:.2f} — clusters partially align with MCF categories.')
    print('  Recommendation: clusters capture orthogonal structure — explore what distinguishes them.')
else:
    print(f'✗ Cluster purity (category): {mean_cat_purity:.2f} — clusters do not align with MCF categories.')
    print('  Recommendation: clusters capture a fundamentally different signal (e.g. seniority, domain blend).')